In [ ]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GENERATOR_PATH = (
    r"C:\Users\hp\Documents\GitHub\Ivhm\BBEK\scripts"
)

if GENERATOR_PATH not in sys.path:
    sys.path.insert(0, GENERATOR_PATH)


from generator import (
    StagedDataGenerator,
    CorrelatedSensorGenerator
)

N_RUNS = 20

N_SENSORS = 100

TOTAL_LENGTH = 20000

OUTPUT_FOLDER = (
    r"C:\Users\hp\Documents\GitHub"
    r"\Ivhm\BBEK\Aircraft_Engine_Data"
)


os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

MASTER_SEED = 42

rng = np.random.default_rng(
    MASTER_SEED
)

sensor_names = [

    f"Sensor{i}"

    for i in range(
        1,
        N_SENSORS + 1
    )

]


print(
    "Number of sensors:",
    len(sensor_names)
)

print(
    "First sensors:",
    sensor_names[:10]
)

print(
    "Last sensors:",
    sensor_names[-10:]
)

PRIMARY_PARAMETER_RANGES = {

    "n_rises": (
        3,
        7
    ),

    "n_falls": (
        2,
        5
    ),

    "value_low": (
        20,
        50
    ),

    "value_high": (
        60,
        120
    ),

    "rise_length_min": (
        10,
        30
    ),

    "rise_length_max": (
        40,
        80
    ),

    "fall_length_min": (
        10,
        30
    ),

    "fall_length_max": (
        30,
        60
    ),

    "stable_length_min": (
        150,
        250
    ),

    "stable_length_max": (
        300,
        500
    ),

    "rise_noise_min": (
        0.05,
        0.20
    ),

    "rise_noise_max": (
        0.20,
        0.50
    ),

    "fall_noise_min": (
        0.005,
        0.02
    ),

    "fall_noise_max": (
        0.02,
        0.08
    ),

    "stable_noise_min": (
        0.005,
        0.02
    ),

    "stable_noise_max": (
        0.02,
        0.08
    ),

    "curvature": (
        0.3,
        1.0
    )

}

def create_sensor_parameters(
    sensor_index
):

    low = (
        10
        + sensor_index * 2
    )


    high = (
        low
        + 50
        + (sensor_index % 10) * 10
    )

    stable_noise_min = rng.uniform(
        0.01,
        0.10
    )

    stable_noise_max = rng.uniform(
        stable_noise_min,
        stable_noise_min + 0.50
    )


    rise_noise_min = rng.uniform(
        0.01,
        0.20
    )

    rise_noise_max = rng.uniform(
        rise_noise_min,
        rise_noise_min + 0.50
    )


    fall_noise_min = rng.uniform(
        0.01,
        0.10
    )

    fall_noise_max = rng.uniform(
        fall_noise_min,
        fall_noise_min + 0.40
    )
    return {

        "new_value_range": (
            low,
            high
        ),

        "stable_noise": (
            stable_noise_min,
            stable_noise_max
        ),

        "rise_noise": (
            rise_noise_min,
            rise_noise_max
        ),

        "fall_noise": (
            fall_noise_min,
            fall_noise_max
        )

    }
def create_run_parameters(
    run_number
):
    run_rng = np.random.default_rng(
        MASTER_SEED + run_number
    )
    
    n_rises = run_rng.integers(

        PRIMARY_PARAMETER_RANGES[
            "n_rises"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "n_rises"
        ][1] + 1

    )

    n_falls = run_rng.integers(

        PRIMARY_PARAMETER_RANGES[
            "n_falls"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "n_falls"
        ][1] + 1

    )

    value_low = run_rng.uniform(

        PRIMARY_PARAMETER_RANGES[
            "value_low"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "value_low"
        ][1]

    )


    value_high = run_rng.uniform(

        PRIMARY_PARAMETER_RANGES[
            "value_high"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "value_high"
        ][1]

    )


    # Make sure high > low

    if value_high <= value_low:

        value_high = value_low + 20


    rise_length_min = run_rng.integers(

        PRIMARY_PARAMETER_RANGES[
            "rise_length_min"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "rise_length_min"
        ][1] + 1

    )


    rise_length_max = run_rng.integers(

        max(
            rise_length_min + 1,
            PRIMARY_PARAMETER_RANGES[
                "rise_length_max"
            ][0]
        ),

        PRIMARY_PARAMETER_RANGES[
            "rise_length_max"
        ][1] + 1

    )


    fall_length_min = run_rng.integers(

        PRIMARY_PARAMETER_RANGES[
            "fall_length_min"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "fall_length_min"
        ][1] + 1

    )


    fall_length_max = run_rng.integers(

        max(
            fall_length_min + 1,
            PRIMARY_PARAMETER_RANGES[
                "fall_length_max"
            ][0]
        ),

        PRIMARY_PARAMETER_RANGES[
            "fall_length_max"
        ][1] + 1

    )


    stable_length_min = run_rng.integers(

        PRIMARY_PARAMETER_RANGES[
            "stable_length_min"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "stable_length_min"
        ][1] + 1

    )


    stable_length_max = run_rng.integers(

        max(
            stable_length_min + 1,
            PRIMARY_PARAMETER_RANGES[
                "stable_length_max"
            ][0]
        ),

        PRIMARY_PARAMETER_RANGES[
            "stable_length_max"
        ][1] + 1

    )

    rise_noise_min = run_rng.uniform(

        PRIMARY_PARAMETER_RANGES[
            "rise_noise_min"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "rise_noise_min"
        ][1]

    )


    rise_noise_max = run_rng.uniform(

        rise_noise_min,

        PRIMARY_PARAMETER_RANGES[
            "rise_noise_max"
        ][1]

    )


    fall_noise_min = run_rng.uniform(

        PRIMARY_PARAMETER_RANGES[
            "fall_noise_min"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "fall_noise_min"
        ][1]

    )


    fall_noise_max = run_rng.uniform(

        fall_noise_min,

        PRIMARY_PARAMETER_RANGES[
            "fall_noise_max"
        ][1]

    )

    stable_noise_min = run_rng.uniform(

        PRIMARY_PARAMETER_RANGES[
            "stable_noise_min"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "stable_noise_min"
        ][1]

    )


    stable_noise_max = run_rng.uniform(

        stable_noise_min,

        PRIMARY_PARAMETER_RANGES[
            "stable_noise_max"
        ][1]

    )

    curvature = run_rng.uniform(

        PRIMARY_PARAMETER_RANGES[
            "curvature"
        ][0],

        PRIMARY_PARAMETER_RANGES[
            "curvature"
        ][1]

    )

    return {

        "n_rises":
        int(n_rises),

        "n_falls":
        int(n_falls),

        "value_range":
        (
            round(value_low, 3),
            round(value_high, 3)
        ),

        "rise_length_bounds":
        (
            int(rise_length_min),
            int(rise_length_max)
        ),

        "fall_length_bounds":
        (
            int(fall_length_min),
            int(fall_length_max)
        ),

        "stable_length_bounds":
        (
            int(stable_length_min),
            int(stable_length_max)
        ),

        "rise_noise":
        (
            round(rise_noise_min, 4),
            round(rise_noise_max, 4)
        ),

        "fall_noise":
        (
            round(fall_noise_min, 4),
            round(fall_noise_max, 4)
        ),

        "stable_noise":
        (
            round(stable_noise_min, 4),
            round(stable_noise_max, 4)
        ),

        "curvature":
        round(curvature, 4),

        "seed":
        MASTER_SEED + run_number

    }

def create_primary_sensor(
    run_number,
    parameters
):

    primary_generator = StagedDataGenerator(

        n_rises=
        parameters["n_rises"],

        n_falls=
        parameters["n_falls"],

        value_range=
        parameters["value_range"],

        total_length=
        TOTAL_LENGTH,

        rise_length_bounds=
        parameters[
            "rise_length_bounds"
        ],

        fall_length_bounds=
        parameters[
            "fall_length_bounds"
        ],

        stable_length_bounds=
        parameters[
            "stable_length_bounds"
        ],

        rise_noise=
        parameters[
            "rise_noise"
        ],

        fall_noise=
        parameters[
            "fall_noise"
        ],

        stable_noise=
        parameters[
            "stable_noise"
        ],

        curvature=
        parameters[
            "curvature"
        ],

        seed=
        parameters["seed"]

    )


    primary_array = (
        primary_generator.generate()
    )

    if len(primary_array) != TOTAL_LENGTH:

        raise ValueError(

            f"Run {run_number}: "
            f"Expected {TOTAL_LENGTH} samples, "
            f"but generator returned "
            f"{len(primary_array)}"

        )


    return (
        primary_generator,
        np.asarray(primary_array)
    )

def save_primary_csv(
    run_number,
    primary_array,
    run_folder
):

    primary_df = pd.DataFrame({

        "Sensor1":
        primary_array

    })


    primary_file = os.path.join(

        run_folder,

        f"Primary_Run{run_number:02d}.csv"

    )


    primary_df.to_csv(

        primary_file,

        index=False

    )


    return primary_file

def create_correlated_sensors(
    primary_generator
):

    sensor_data = {}

    primary_array = (
        primary_generator.generate()
    )


    sensor_data[
        "Sensor1"
    ] = primary_array

    for sensor_index in range(
        2,
        N_SENSORS + 1
    ):

        parameters = (
            create_sensor_parameters(
                sensor_index
            )
        )


        sensor_generator = (
            CorrelatedSensorGenerator(

                reference_generator=
                primary_generator,

                new_value_range=
                parameters[
                    "new_value_range"
                ],

                stable_noise=
                parameters[
                    "stable_noise"
                ],

                rise_noise=
                parameters[
                    "rise_noise"
                ],

                fall_noise=
                parameters[
                    "fall_noise"
                ]

            )
        )


        values = (
            sensor_generator.generate()
        )


        if len(values) != TOTAL_LENGTH:

            raise ValueError(

                f"Sensor{sensor_index}: "
                f"Expected {TOTAL_LENGTH} "
                f"samples, got {len(values)}"

            )


        sensor_data[
            f"Sensor{sensor_index}"
        ] = values


    return sensor_data

def save_run_csv(
    run_number,
    sensor_data,
    run_folder
):

    df = pd.DataFrame(
        sensor_data
    )


    # Make sure column order is exactly Sensor1 ... Sensor100

    df = df[
        sensor_names
    ]


    output_file = os.path.join(

        run_folder,

        f"Run{run_number:02d}_AllSensors.csv"

    )


    df.to_csv(

        output_file,

        index=False

    )


    return output_file

def save_run_parameters(
    all_run_parameters
):

    parameter_rows = []


    for run_number, p in (
        all_run_parameters.items()
    ):

        parameter_rows.append({

            "Run":
            run_number,

            "n_rises":
            p["n_rises"],

            "n_falls":
            p["n_falls"],

            "value_range_low":
            p["value_range"][0],

            "value_range_high":
            p["value_range"][1],

            "rise_length_min":
            p["rise_length_bounds"][0],

            "rise_length_max":
            p["rise_length_bounds"][1],

            "fall_length_min":
            p["fall_length_bounds"][0],

            "fall_length_max":
            p["fall_length_bounds"][1],

            "stable_length_min":
            p["stable_length_bounds"][0],

            "stable_length_max":
            p["stable_length_bounds"][1],

            "rise_noise_min":
            p["rise_noise"][0],

            "rise_noise_max":
            p["rise_noise"][1],

            "fall_noise_min":
            p["fall_noise"][0],

            "fall_noise_max":
            p["fall_noise"][1],

            "stable_noise_min":
            p["stable_noise"][0],

            "stable_noise_max":
            p["stable_noise"][1],

            "curvature":
            p["curvature"],

            "seed":
            p["seed"]

        })


    df = pd.DataFrame(
        parameter_rows
    )


    parameter_file = os.path.join(

        OUTPUT_FOLDER,

        "run_parameters.csv"

    )


    df.to_csv(

        parameter_file,

        index=False

    )


    return parameter_file

all_run_parameters = {}


print("\n")
print("=" * 70)
print("AIRCRAFT ENGINE DATA GENERATION")
print("=" * 70)


for run_number in range(
    1,
    N_RUNS + 1
):


    print("\n")
    print(
        "-" * 70
    )

    print(
        f"GENERATING RUN {run_number}/{N_RUNS}"
    )

    print(
        "-" * 70
    )

    run_folder = os.path.join(

        OUTPUT_FOLDER,

        f"Run{run_number:02d}"

    )


    os.makedirs(
        run_folder,
        exist_ok=True
    )

    parameters = (
        create_run_parameters(
            run_number
        )
    )


    all_run_parameters[
        run_number
    ] = parameters

    print(
        "n_rises       :",
        parameters["n_rises"]
    )

    print(
        "n_falls       :",
        parameters["n_falls"]
    )

    print(
        "value_range   :",
        parameters["value_range"]
    )

    print(
        "rise_noise    :",
        parameters["rise_noise"]
    )

    print(
        "fall_noise    :",
        parameters["fall_noise"]
    )

    print(
        "stable_noise  :",
        parameters["stable_noise"]
    )

    print(
        "curvature     :",
        parameters["curvature"]
    )

    (
        primary_generator,
        primary_array

    ) = create_primary_sensor(

        run_number,

        parameters

    )

    primary_file = (
        save_primary_csv(

            run_number,

            primary_array,

            run_folder

        )
    )


    print(
        "Primary CSV:",
        primary_file
    )

    sensor_data = (
        create_correlated_sensors(

            primary_generator

        )
    )

    all_sensor_file = (
        save_run_csv(

            run_number,

            sensor_data,

            run_folder

        )
    )


    print(
        "100-sensor CSV:",
        all_sensor_file
    )

    print(
        "Rows:",
        len(sensor_data["Sensor1"])
    )

    print(
        "Columns:",
        len(sensor_data)
    )

parameter_file = (
    save_run_parameters(
        all_run_parameters
    )
)

print("\n")
print("=" * 70)

print(
    "DATA GENERATION COMPLETED"
)

print("=" * 70)

print(
    "Number of runs       :",
    N_RUNS
)

print(
    "Sensors per run      :",
    N_SENSORS
)

print(
    "Samples per run      :",
    TOTAL_LENGTH
)

print(
    "Primary sensors      :",
    N_RUNS
)

print(
    "Full sensor CSVs     :",
    N_RUNS
)

print(
    "Parameter file       :",
    parameter_file
)

print(
    "Output folder        :",
    OUTPUT_FOLDER
)

print("=" * 70)

Number of sensors: 100
First sensors: ['Sensor1', 'Sensor2', 'Sensor3', 'Sensor4', 'Sensor5', 'Sensor6', 'Sensor7', 'Sensor8', 'Sensor9', 'Sensor10']
Last sensors: ['Sensor91', 'Sensor92', 'Sensor93', 'Sensor94', 'Sensor95', 'Sensor96', 'Sensor97', 'Sensor98', 'Sensor99', 'Sensor100']


AIRCRAFT ENGINE DATA GENERATION


----------------------------------------------------------------------
GENERATING RUN 1/20
----------------------------------------------------------------------
n_rises       : 5
n_falls       : 4
value_range   : (21.313, 61.202)
rise_noise    : (0.1628, 0.2517)
fall_noise    : (0.0113, 0.0423)
stable_noise  : (0.0193, 0.0734)
curvature     : 0.495
Primary CSV: C:\Users\hp\Documents\GitHub\Ivhm\BBEK\Aircraft_Engine_Data\Run01\Primary_Run01.csv
100-sensor CSV: C:\Users\hp\Documents\GitHub\Ivhm\BBEK\Aircraft_Engine_Data\Run01\Run01_AllSensors.csv
Rows: 20000
Columns: 100


----------------------------------------------------------------------
GENERATING RUN 2/20
--------